# SecureCode AI — CodeBERT Fine-Tuning for Binary Vulnerability Detection

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/0Tusharbhardwaj/nlp/blob/main/notebooks/01_codebert_finetuning.ipynb)
[![GitHub Repository](https://img.shields.io/badge/GitHub-0Tusharbhardwaj%2Fnlp-blue.svg)](https://github.com/0Tusharbhardwaj/nlp)

## Scientific Objective
Fine-tune **Microsoft CodeBERT** (`microsoft/codebert-base`) for function-level binary vulnerability detection on the **CodeXGLUE / Devign** benchmark.

- **Class 0:** Non-vulnerable (Safe)
- **Class 1:** Vulnerable (Security Defect)
- **Evaluation Focus:** F1-score, False Negative Rate (FNR = 1 - Recall), ROC-AUC, and Confusion Matrix.
- **Rigor:** Validation split is used for model selection; the test set remains untouched until final scoring.

### Step 1: Hardware Acceleration & GPU Check
Verify NVIDIA GPU availability, CUDA version, and tensor cores.

In [ ]:
!nvidia-smi

import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
    print(f"Device Count: {torch.cuda.device_count()}")
    device = torch.device("cuda")
else:
    print("WARNING: No GPU detected. Training CodeBERT on CPU will be slow.")
    device = torch.device("cpu")

### Step 2: Mount Google Drive & Environment Setup
Mount Google Drive to persist checkpoints and final models across Colab runtime timeouts.

In [ ]:
import os
from pathlib import Path

# Check if running inside Google Colab
try:
    from google.colab import drive
    drive.mount('/content/drive')
    drive_dir = Path('/content/drive/MyDrive/SecureCode-AI')
    drive_dir.mkdir(parents=True, exist_ok=True)
    print(f"Google Drive mounted successfully at: {drive_dir}")
except ImportError:
    drive_dir = Path('results')
    drive_dir.mkdir(parents=True, exist_ok=True)
    print("Running locally or in non-Colab environment. Checkpoints will save to ./results")

### Step 3: Clone Repository & Install Dependencies

In [ ]:
# Pull latest repository updates or clone fresh
import os
from pathlib import Path

if Path('nlp').exists():
    %cd nlp
    !git pull origin main
elif Path('datasets').exists():
    !git pull origin main
else:
    !git clone https://github.com/0Tusharbhardwaj/nlp.git
    %cd nlp

print(f'Current Working Directory: {os.getcwd()}')

# Install dependencies compatible with Python 3.10/3.11 in Colab
!pip install -q transformers datasets accelerate scikit-learn pandas matplotlib seaborn pyyaml


### Step 4: Load Validated Preprocessed Dataset & Display Class Statistics

In [ ]:
import os
import json
import subprocess
from pathlib import Path
from collections import Counter

# Verify directory is inside repository root
if Path('nlp').exists() and not Path('datasets').exists():
    os.chdir('nlp')

# Auto-run preprocessing pipeline if processed partition is missing
train_path = Path('datasets/processed/devign/train.jsonl')
if not train_path.exists():
    print('[!] Processed dataset not found. Running reproducible preprocessing pipeline...')
    subprocess.run(['python', 'scripts/preprocess_data.py'], check=True)

def load_split(split_name):
    path = f'datasets/processed/devign/{split_name}.jsonl'
    samples = []
    with open(path, 'r', encoding='utf-8') as f:
        for line in f:
            if line.strip():
                samples.append(json.loads(line))
    return samples

train_data = load_split('train')
valid_data = load_split('valid')
test_data = load_split('test')

print(f'Dataset Splits Loaded:')
print(f'  - Train: {len(train_data)} samples')
print(f'  - Valid: {len(valid_data)} samples')
print(f'  - Test:  {len(test_data)} samples')

train_labels = [s['label'] for s in train_data]
counts = Counter(train_labels)
print(f'\nTrain Class Distribution: Safe (0): {counts[0]} | Vulnerable (1): {counts[1]}')
imbalance = counts[0] / max(counts[1], 1)
print(f'Class Imbalance Ratio (0 to 1): {imbalance:.2f}')


### Step 5: CodeBERT Tokenizer & Untruncated Token Length Audit
We inspect token length distributions **before** applying any sequence truncation.

In [ ]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("microsoft/codebert-base", use_fast=True)

# Compute untruncated lengths
all_codes = [s["codebert_code"] for s in train_data + valid_data + test_data]
untruncated_lengths = [len(tokenizer.encode(code, truncation=False)) for code in all_codes]

print("=== Untruncated Token Length Statistics ===")
print(f"Total Samples: {len(untruncated_lengths)}")
print(f"Min Tokens:    {np.min(untruncated_lengths)}")
print(f"Max Tokens:    {np.max(untruncated_lengths)}")
print(f"Mean Tokens:   {np.mean(untruncated_lengths):.1f}")
print(f"Median Tokens: {np.median(untruncated_lengths):.1f}")
print(f"P90 Tokens:    {np.percentile(untruncated_lengths, 90):.1f}")
print(f"P95 Tokens:    {np.percentile(untruncated_lengths, 95):.1f}")

exceeding_512 = sum(1 for l in untruncated_lengths if l > 512)
print(f"Samples Exceeding 512 Window: {exceeding_512} ({exceeding_512 / len(untruncated_lengths) * 100:.1f}%)")

### Step 6: Initialize CodeBERT Sequence Classification Model
We instantiate `microsoft/codebert-base` with a 2-label classification head.

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "microsoft/codebert-base",
    num_labels=2,
)
model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total Parameters:     {total_params:,}")
print(f"Trainable Parameters: {trainable_params:,}")

### Step 7: PyTorch Dataset, DataLoaders & Class-Weighted Loss

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

class VulnerabilityDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

    def __len__(self):
        return len(self.labels)

MAX_LEN = 512
BATCH_SIZE = 16

train_enc = tokenizer([s["codebert_code"] for s in train_data], padding="max_length", truncation=True, max_length=MAX_LEN)
valid_enc = tokenizer([s["codebert_code"] for s in valid_data], padding="max_length", truncation=True, max_length=MAX_LEN)
test_enc  = tokenizer([s["codebert_code"] for s in test_data],  padding="max_length", truncation=True, max_length=MAX_LEN)

train_ds = VulnerabilityDataset(train_enc, [s["label"] for s in train_data])
valid_ds = VulnerabilityDataset(valid_enc, [s["label"] for s in valid_data])
test_ds  = VulnerabilityDataset(test_enc,  [s["label"] for s in test_data])

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
valid_loader = DataLoader(valid_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False)

# Class-weighted CrossEntropy loss
n_total = len(train_data)
n_0 = counts[0]
n_1 = counts[1]
weights = torch.tensor([n_total / (2.0 * max(n_0, 1)), n_total / (2.0 * max(n_1, 1))], dtype=torch.float, device=device)
criterion = nn.CrossEntropyLoss(weight=weights)
print(f"Class weights configured: {weights}")

### Step 8: Training Configuration, Optimizer & Checkpoint Recovery Setup
Includes automated session recovery so interrupted Colab sessions resume without starting over.

In [ ]:
from transformers import get_linear_schedule_with_warmup

EPOCHS = 3
LEARNING_RATE = 2e-5
USE_FP16 = torch.cuda.is_available()

optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=0.01)
total_steps = len(train_loader) * EPOCHS
warmup_steps = int(total_steps * 0.1)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps)
scaler = torch.cuda.amp.GradScaler(enabled=USE_FP16)

checkpoint_path = drive_dir / "checkpoints" / "codebert_best.pt"
checkpoint_path.parent.mkdir(parents=True, exist_ok=True)

start_epoch = 0
best_val_f1 = -1.0
history = []

if checkpoint_path.exists():
    print(f"Found existing checkpoint at {checkpoint_path}. Resuming session...")
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    start_epoch = ckpt.get("epoch", 0) + 1
    best_val_f1 = ckpt.get("best_f1", -1.0)
    history = ckpt.get("history", [])
    print(f"Resumed at Epoch {start_epoch} with Best F1: {best_val_f1:.4f}")
else:
    print("Starting fresh training session.")

### Step 9: Fine-Tuning Execution & Validation Evaluation Loop

In [ ]:
from src.evaluation import compute_vulnerability_metrics

print(f"Beginning fine-tuning ({start_epoch} to {EPOCHS} epochs)...")

for epoch in range(start_epoch, EPOCHS):
    model.train()
    total_loss = 0.0

    for batch in train_loader:
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        with torch.cuda.amp.autocast(enabled=USE_FP16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            loss = criterion(outputs.logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        total_loss += loss.item()

    avg_loss = total_loss / max(len(train_loader), 1)

    # Validation Evaluation
    model.eval()
    val_preds, val_probs, val_true = [], [], []
    with torch.no_grad():
        for batch in valid_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            with torch.cuda.amp.autocast(enabled=USE_FP16):
                outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
            preds = np.argmax(probs, axis=-1)
            val_probs.extend(probs[:, 1].tolist())
            val_preds.extend(preds.tolist())
            val_true.extend(batch["labels"].tolist())

    val_metrics = compute_vulnerability_metrics(val_true, val_preds, val_probs)
    val_f1 = val_metrics["f1_score"]
    val_fnr = val_metrics["false_negative_rate"]

    print(f"Epoch {epoch+1}/{EPOCHS} | Train Loss: {avg_loss:.4f} | Val F1: {val_f1:.4f} | Val FNR: {val_fnr:.4f} | Val Acc: {val_metrics['accuracy']:.4f}")

    history.append({"epoch": epoch + 1, "loss": avg_loss, "val_f1": val_f1, "val_fnr": val_fnr})

    # Checkpoint if best validation F1
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        print(f"[*] Best model improved! Saving checkpoint to {checkpoint_path}")
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_f1": best_val_f1,
            "history": history,
        }, checkpoint_path)
        # Save Hugging Face artifacts
        model.save_pretrained(str(drive_dir / "models" / "codebert_finetuned"))
        tokenizer.save_pretrained(str(drive_dir / "models" / "codebert_finetuned"))

### Step 10: Untouched Test Set Evaluation & Scientific Reporting
The test set is evaluated **only after** model selection is finalized.

In [ ]:
# Reload best checkpoint
if checkpoint_path.exists():
    ckpt = torch.load(checkpoint_path, map_location=device)
    model.load_state_dict(ckpt["model_state_dict"])
    print(f"Loaded best model from validation check (Best Val F1: {ckpt.get('best_f1'):.4f})")

model.eval()
test_preds, test_probs, test_true = [], [], []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        with torch.cuda.amp.autocast(enabled=USE_FP16):
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()
        preds = np.argmax(probs, axis=-1)
        test_probs.extend(probs[:, 1].tolist())
        test_preds.extend(preds.tolist())
        test_true.extend(batch["labels"].tolist())

final_test_metrics = compute_vulnerability_metrics(test_true, test_preds, test_probs)

print("\n================ FINAL CODEBERT TEST RESULTS ================")
print(f"Accuracy:              {final_test_metrics['accuracy']:.4f}")
print(f"Precision:             {final_test_metrics['precision']:.4f}")
print(f"Recall:                {final_test_metrics['recall']:.4f}")
print(f"F1-Score:              {final_test_metrics['f1_score']:.4f}")
print(f"False Negative Rate:   {final_test_metrics['false_negative_rate']:.4f}")
print(f"ROC-AUC:               {final_test_metrics['roc_auc']}")
print(f"Confusion Matrix:      {final_test_metrics['confusion_matrix']}")
print("=============================================================")

### Step 11: Plot Confusion Matrix & Save Metrics to Google Drive

In [ ]:
from src.evaluation import plot_confusion_matrix

fig_path = drive_dir / "figures" / "codebert_colab_cm.png"
fig_path.parent.mkdir(parents=True, exist_ok=True)
plot_confusion_matrix(
    final_test_metrics["confusion_matrix"],
    title="CodeBERT Fine-Tuned (Devign Test Split)",
    output_path=fig_path,
    cmap="Purples",
)
print(f"Confusion matrix figure saved to: {fig_path}")

# Save metrics JSON (ensuring parent directory exists)
metrics_path = drive_dir / "metrics" / "codebert_colab_metrics.json"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
with open(metrics_path, "w", encoding="utf-8") as f:
    json.dump({
        "model": "microsoft/codebert-base",
        "test_metrics": final_test_metrics,
        "training_history": history,
        "hyperparameters": {
            "epochs": EPOCHS,
            "learning_rate": LEARNING_RATE,
            "batch_size": BATCH_SIZE,
            "max_seq_length": MAX_LEN,
            "seed": 42,
        }
    }, f, indent=2)
print(f"Metrics saved to: {metrics_path}")


### Step 12: Interactive Single-Snippet Vulnerability Scanner Demo

In [ ]:
sample_code = """
static void unsafe_copy(char *dest, const char *src) {
    // Unchecked string copy prone to buffer overflow
    strcpy(dest, src);
}
"""

inputs = tokenizer(sample_code, return_tensors="pt", truncation=True, max_length=512).to(device)
model.eval()
with torch.no_grad():
    logits = model(**inputs).logits
    probs = torch.softmax(logits, dim=-1).cpu().numpy()[0]

print("Scan Result:")
print(f"  Classification: {'VULNERABLE (Class 1)' if probs[1] >= 0.5 else 'SAFE (Class 0)'}")
print(f"  Confidence / Probability of Vulnerability: {probs[1]:.4f}")
print(f"  Probability of Safe: {probs[0]:.4f}")